# 1C 코어온도 추정 가능성 평가

실제 Daly 1C 방전 로그를 입력으로 사용하고, 기계과 열해석 `T_core`를 정답 라벨로 사용한다. 충전 데이터는 제외한다.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
P=Path(r"C:\Users\YechanLee\Claude\Projects\인종설경"); L=Path(r"C:\Users\YechanLee\Documents\카카오톡 받은 파일\T_data_1sec_interpolated.xlsx"); O=Path(r"C:\jupyter\notebook\ml_validation\output\core_temperature_1C"); O.mkdir(parents=True,exist_ok=True)
lab=pd.read_excel(L).rename(columns={"Time":"time_s","T_core":"tcore_c","T_surface":"tsurface_c"})
features=["physical_current_a","pack_v","avg_cell_v","cell_delta_mv","temp1_c","temp2_c"]
def load(p):
 d=pd.read_csv(p,encoding="utf-8-sig"); cs=["elapsed_s","pack_v","current_a","temp1_c","temp2_c","cell_delta_mv"]; d[cs]=d[cs].apply(pd.to_numeric,errors="coerce"); cells=[c for c in d if c.startswith("cell") and c.endswith("_v") and c[4:-2].isdigit()]; d["avg_cell_v"]=d[cells].mean(axis=1).fillna(d.pack_v/4); a=d[d.current_a.abs()>.05]
 if len(a)<300 or a.pack_v.iloc[-1]>=a.pack_v.iloc[0]-.05:return None
 d=d.loc[d.elapsed_s>=a.elapsed_s.iloc[0]].copy().reset_index(drop=True); d["physical_current_a"]=d.current_a if a.current_a.mean()>0 else -d.current_a; d["t_rel_s"]=d.elapsed_s-d.elapsed_s.iloc[0]; d["file"]=p.name
 return d if d.t_rel_s.iloc[-1]>=2800 else None
logs=[d for p in sorted(P.glob("daly_log_*.csv")) if (d:=load(p)) is not None]
for d in logs:
 t=np.clip(d.t_rel_s,0,lab.time_s.iloc[-1]); d["tcore_label_c"]=np.interp(t,lab.time_s,lab.tcore_c)+(d.temp1_c-25); d["tsurface_label_c"]=np.interp(t,lab.time_s,lab.tsurface_c)+(d.temp1_c-25)
print([(d.file,round(d.t_rel_s.iloc[-1]),round(d.physical_current_a.mean(),2)) for d in logs])


In [ ]:
train=[d for d in logs if d.file in ["daly_log_20260827_191847.csv","daly_log_20260828_170236.csv"]]; test=[d for d in logs if d.file=="daly_log_20260829_165601.csv"]; sx=StandardScaler().fit(pd.concat([d[features] for d in train])); sy=StandardScaler().fit(pd.concat([d[["tcore_label_c"]] for d in train])); LOOK=60; H=12
def arr(ds):
 X=[]; y=[]; tm=[]
 for d in ds:
  x=sx.transform(d[features].interpolate(limit=2).ffill().bfill()); z=sy.transform(d[["tcore_label_c"]]).ravel()
  for j in range(LOOK,len(d)-H,5): X.append(x[j-LOOK:j]); y.append(z[j+H-1]); tm.append(d.t_rel_s.iloc[j+H-1])
 return np.asarray(X,np.float32),np.asarray(y,np.float32),tm
Xtr,ytr,_=arr(train); Xte,yte,tm=arr(test); print("train",Xtr.shape,"test",Xte.shape)
class Net(nn.Module):
 def __init__(self):
  super().__init__(); self.r=nn.LSTM(len(features),48,2,batch_first=True,dropout=.15); self.f=nn.Sequential(nn.Linear(48,24),nn.ReLU(),nn.Linear(24,1))
 def forward(self,x):return self.f(self.r(x)[0][:,-1]).squeeze(-1)
device="cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(42); m=Net().to(device); opt=torch.optim.Adam(m.parameters(),lr=2e-3); lf=nn.MSELoss(); dl=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=512,shuffle=True); xt=torch.tensor(Xte,device=device); yt=torch.tensor(yte,device=device); hist=[]; best=(1e9,None)
for ep in range(1,21):
 m.train(); q=[]
 for xb,yb in dl:
  xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); v=lf(m(xb),yb); v.backward(); opt.step(); q.append(v.item())
 m.eval(); val=lf(m(xt),yt).item(); hist.append([ep,np.mean(q),val]);
 if val<best[0]:best=(val,{k:v.detach().cpu().clone() for k,v in m.state_dict().items()})
 if ep==1 or ep%5==0:print(ep,np.mean(q),val)
m.load_state_dict(best[1]); torch.save(m.state_dict(),O/"core_temperature_lstm.pt")


In [ ]:
m.eval(); pred=sy.inverse_transform(m(xt).detach().cpu().numpy()[:,None]).ravel(); actual=sy.inverse_transform(yte[:,None]).ravel(); proxy=sy.inverse_transform(Xte[:,-1,features.index("temp2_c")][:,None]).ravel(); metrics=pd.DataFrame([{"model":"T2 proxy","MAE_C":mean_absolute_error(actual,proxy),"RMSE_C":mean_squared_error(actual,proxy)**.5,"R2":r2_score(actual,proxy)},{"model":"LSTM Tcore","MAE_C":mean_absolute_error(actual,pred),"RMSE_C":mean_squared_error(actual,pred)**.5,"R2":r2_score(actual,pred)}]); display(metrics.round(4)); metrics.to_csv(O/"metrics_overall.csv",index=False); result=pd.DataFrame({"time_s":tm,"tcore_label_c":actual,"lstm_tcore_c":pred,"t2_proxy_c":proxy}); result.to_csv(O/"test_predictions.csv",index=False); plt.figure(figsize=(12,5)); plt.plot(result.time_s/60,result.tcore_label_c,label="mechanical Tcore"); plt.plot(result.time_s/60,result.lstm_tcore_c,"--",label="LSTM"); plt.plot(result.time_s/60,result.t2_proxy_c,":",label="T2 proxy"); plt.legend(); plt.grid(alpha=.25); plt.xlabel("time [min]"); plt.ylabel("temperature [C]"); plt.title("Independent 1C discharge"); plt.tight_layout(); plt.savefig(O/"01_1C_core_prediction.png",dpi=160); plt.show()


## 판정
LSTM이 T2 proxy보다 작게 오차가 나오면 1C 조건에서 웹 대시보드 적용 가능성을 검토한다. 단, 실제 코어 센서 검증 전까지는 열해석 Tcore 라벨 추정 결과로 표현한다.